# API Model Registration Demo

This notebook walks through registering a model that is served via an external PyMC API.
The API is packaged as a Docker service (`pymc_api`) which exposes a synchronous `/runs`
endpoint. The Model Zoo registry stores the API binding and uses it when executing runs.

## 1. Start the Docker services

The command below launches Postgres, the registry API, and the dummy PyMC service.
Skip this step if you already have them running.

In [1]:
%%bash
set -euxo pipefail
docker compose up -d postgres api pymc_api
docker compose ps

+ docker compose up -d postgres api pymc_api
time="2025-10-16T10:01:54+05:30" level=warning msg="/Users/subhagatoadak/Documents/GitHub/model-zoo/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container model-zoo-postgres  Running
 Container model-zoo-pymc-api  Running
 Container model-zoo-api  Running
 Container model-zoo-postgres  Waiting
 Container model-zoo-postgres  Healthy
+ docker compose ps
time="2025-10-16T10:01:55+05:30" level=warning msg="/Users/subhagatoadak/Documents/GitHub/model-zoo/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"


NAME                 IMAGE                COMMAND                  SERVICE    CREATED         STATUS                   PORTS
model-zoo-api        model-zoo-api        "uvicorn model_zoo.a…"   api        9 minutes ago   Up 5 minutes             0.0.0.0:8000->8000/tcp
model-zoo-postgres   postgres:15          "docker-entrypoint.s…"   postgres   44 hours ago    Up 6 minutes (healthy)   0.0.0.0:5432->5432/tcp
model-zoo-pymc-api   model-zoo-pymc_api   "uvicorn app:app --h…"   pymc_api   16 hours ago    Up 7 minutes             0.0.0.0:8100->8100/tcp


## 2. Inspect the API-backed configuration

The configuration declares `kind: "api"` and binds to the service endpoint.

In [3]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

config_path = (PROJECT_ROOT / 'configs' / 'pymc_api_linear.json').resolve()
api_config = json.loads(config_path.read_text())
config_path, api_config


(PosixPath('/Users/subhagatoadak/Documents/GitHub/model-zoo/configs/pymc_api_linear.json'),
 {'model': {'id': 'pymc_api_linear_v1',
   'name': 'PyMC API Linear Regression',
   'version': '1.0.0',
   'kind': 'api',
   'binding': {'kind': 'api',
    'endpoint': 'http://pymc_api:8100/runs',
    'method': 'POST'},
   'inputs': ['X', 'y']},
  'hyperparameters': {'schema': [{'name': 'scale',
     'dtype': 'float',
     'default': 1.0,
     'description': 'Placeholder hyperparameter'}],
   'values': [{'model_id': 'pymc_api_linear_v1',
     'param': 'scale',
     'value': '1.0'}]},
  'variables': {'definitions': [{'name': 'X', 'source': 'x_values'},
    {'name': 'y', 'source': 'y_values'}]},
  'context': {'timeout_s': 120}})

## 3. Register the configuration

POST the config to the registry. The binding is stored with the model spec so future runs
can resolve the API endpoint automatically.

In [4]:
import requests

registry_url = 'http://localhost:8000/registry/configs'
response = requests.post(registry_url, json={'env': 'demo-api', 'config': api_config}, timeout=30)
response.raise_for_status()
response.json()

{'model_id': 'pymc_api_linear_v1',
 'environment': 'demo-api',
 'spec': {'model_id': 'pymc_api_linear_v1',
  'name': 'PyMC API Linear Regression',
  'version': '1.0.0',
  'kind': 'api',
  'hyperparams_schema': {'type': 'object',
   'properties': {'scale': {'title': 'scale',
     'description': 'Placeholder hyperparameter',
     'type': 'number',
     'default': 1.0}},
   'required': [],
   'additionalProperties': True},
  'tags': [],
  'metadata': {},
  'dependencies': []},
 'stored': True}

## 4. Submit a run

Synthetic data is sent to the registry. It forwards the inputs to the PyMC API
and returns the predictions and diagnostics.

In [5]:
import numpy as np

x_values = (np.linspace(0.0, 4.5, num=25) + np.random.normal(0, 0.1, size=25)).round(3).tolist()
y_values = [1.5 + 2.2 * x for x in x_values]

run_payload = {
    'model_id': api_config['model']['id'],
    'env': 'demo-api',
    'data_sources': {'x_values': x_values, 'y_values': y_values},
    'context': {'timeout_s': 120}
}

submit_resp = requests.post('http://localhost:8000/runs', json=run_payload, timeout=30)
submit_resp.raise_for_status()
ticket = submit_resp.json()
ticket

{'run_id': 'c5fbd787-cc17-4012-8e51-4648909ed9a4', 'status': 'queued'}

## 5. Poll the run result

The API-backed run returns almost immediately. Inspect the summary returned by the PyMC service.

In [6]:
run_id = ticket['run_id']
status_url = f'http://localhost:8000/runs/{run_id}'
result_resp = requests.get(status_url, timeout=30)
result_resp.raise_for_status()
result = result_resp.json()
result['result']['summary']


{'Call': {'model_id': 'pymc_api_linear_v1', 'params': {}},
 'Coefficients': {'alpha_map': 1.4990266610948788,
  'beta_map': 2.200030145643492},
 'Fit': {'r2': 0.999999906708276, 'sigma_map': 0.0016303963457186329},
 'Residuals': {'rss': 2.0503303320093587e-05}}